# FastAPI Model Serving

In this notebook, the registered uplift model is prepared for real-time inference through a REST API.

The objective is to expose the MLflow `champion` model through a FastAPI service so that external applications can submit customer feature values and receive an uplift prediction and treatment recommendation.

The main objectives are:

- Load the champion T-Learner from the MLflow Model Registry.
- Define a validated API request schema for the 12 model features.
- Generate uplift predictions through the registered model.
- Convert predicted uplift into a treatment recommendation.
- Create API endpoints for health checking and model inference.
- Validate the API logic before running the web server.

The resulting architecture is:

Client → FastAPI → MLflow Model Registry → Champion T-Learner → Uplift Prediction → Treatment Recommendation

This stage moves the project from offline model experimentation toward a deployable machine learning inference service.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import mlflow
import mlflow.pyfunc

from fastapi import FastAPI
from pydantic import BaseModel

print("MLflow:", mlflow.__version__)
print("FastAPI:", __import__("fastapi").__version__)

C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:151: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:186: UserWarning: Field name "schema" shadows an attribute in parent "BaseModel"; 
  warnings.warn(


MLflow: 3.16.0
FastAPI: 0.110.0


In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling"
)

MLFLOW_DIR = (
    PROJECT_ROOT / "mlruns"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nMLflow directory:")
print(MLFLOW_DIR)

Project root:
C:\Users\ugand\customer-churn-uplift-modeling

MLflow directory:
C:\Users\ugand\customer-churn-uplift-modeling\mlruns


In [3]:
mlflow_tracking_path = (
    MLFLOW_DIR / "mlflow.db"
)

mlflow.set_tracking_uri(
    f"sqlite:///{mlflow_tracking_path}"
)

print(
    "MLflow tracking URI:"
)

print(
    mlflow.get_tracking_uri()
)

MLflow tracking URI:
sqlite:///C:\Users\ugand\customer-churn-uplift-modeling\mlruns\mlflow.db


In [4]:
REGISTERED_MODEL_NAME = (
    "customer_uplift_t_learner"
)

MODEL_ALIAS = "champion"

MODEL_URI = (
    f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
)

print("Registered model:")
print(REGISTERED_MODEL_NAME)

print("\nModel alias:")
print(MODEL_ALIAS)

print("\nModel URI:")
print(MODEL_URI)

Registered model:
customer_uplift_t_learner

Model alias:
champion

Model URI:
models:/customer_uplift_t_learner@champion


In [5]:
model = mlflow.pyfunc.load_model(
    MODEL_URI
)

print(
    "Champion model loaded successfully."
)

print(
    "Model type:",
    type(model)
)

Champion model loaded successfully.
Model type: <class 'mlflow.pyfunc.PyFuncModel'>


In [6]:
FEATURE_COLUMNS = [
    f"f{i}"
    for i in range(12)
]

print(
    "Model features:"
)

print(
    FEATURE_COLUMNS
)

Model features:
['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']


In [7]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "criteo-research-uplift-v2.1.csv.gz"
)

test_data = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    nrows=10
)

X_test = test_data[
    FEATURE_COLUMNS
]

predictions = model.predict(
    X_test
)

print(
    "Predictions generated:",
    len(predictions)
)

print(
    "\nPredictions:"
)

print(
    predictions
)

Predictions generated: 10

Predictions:
[-0.00407933 -0.00407933 -0.00407933 -0.00407933 -0.01353526 -0.00407933
 -0.01032156 -0.00407933 -0.01353526 -0.00407933]


In [8]:
predictions = np.asarray(
    predictions
).reshape(-1)

print(
    "Prediction shape:",
    predictions.shape
)

print(
    "First prediction:",
    predictions[0]
)

Prediction shape: (10,)
First prediction: -0.004079329291847572


In [9]:
def get_treatment_recommendation(
    uplift: float
) -> str:

    if uplift > 0:
        return "TREAT"

    return "DO_NOT_TREAT"

In [10]:
for uplift in predictions[:10]:

    recommendation = (
        get_treatment_recommendation(
            float(uplift)
        )
    )

    print(
        f"Uplift: {uplift:.6f} "
        f"→ {recommendation}"
    )

Uplift: -0.004079 → DO_NOT_TREAT
Uplift: -0.004079 → DO_NOT_TREAT
Uplift: -0.004079 → DO_NOT_TREAT
Uplift: -0.004079 → DO_NOT_TREAT
Uplift: -0.013535 → DO_NOT_TREAT
Uplift: -0.004079 → DO_NOT_TREAT
Uplift: -0.010322 → DO_NOT_TREAT
Uplift: -0.004079 → DO_NOT_TREAT
Uplift: -0.013535 → DO_NOT_TREAT
Uplift: -0.004079 → DO_NOT_TREAT


In [11]:
app = FastAPI(
    title="Customer Uplift Modeling API",
    description=(
        "API for real-time uplift prediction "
        "using the MLflow champion T-Learner."
    ),
    version="1.0.0"
)

print(
    "FastAPI application created."
)

FastAPI application created.


In [12]:
class UpliftRequest(BaseModel):

    f0: float
    f1: float
    f2: float
    f3: float
    f4: float
    f5: float
    f6: float
    f7: float
    f8: float
    f9: float
    f10: float
    f11: float

In [13]:
class UpliftResponse(BaseModel):

    predicted_uplift: float
    recommendation: str
    model_name: str
    model_alias: str

C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:151: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:151: UserWarning: Field "model_alias" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [14]:
@app.get("/health")
def health_check():

    return {
        "status": "healthy",
        "model": REGISTERED_MODEL_NAME,
        "alias": MODEL_ALIAS
    }

In [15]:
@app.post(
    "/predict",
    response_model=UpliftResponse
)
def predict_uplift(
    request: UpliftRequest
):

    input_data = pd.DataFrame(
        [
            {
                feature: getattr(
                    request,
                    feature
                )
                for feature in FEATURE_COLUMNS
            }
        ]
    )

    prediction = model.predict(
        input_data
    )

    uplift = float(
        np.asarray(
            prediction
        ).reshape(-1)[0]
    )

    recommendation = (
        get_treatment_recommendation(
            uplift
        )
    )

    return UpliftResponse(
        predicted_uplift=uplift,
        recommendation=recommendation,
        model_name=REGISTERED_MODEL_NAME,
        model_alias=MODEL_ALIAS
    )

In [16]:
sample_request = UpliftRequest(
    f0=float(X_test.iloc[0]["f0"]),
    f1=float(X_test.iloc[0]["f1"]),
    f2=float(X_test.iloc[0]["f2"]),
    f3=float(X_test.iloc[0]["f3"]),
    f4=float(X_test.iloc[0]["f4"]),
    f5=float(X_test.iloc[0]["f5"]),
    f6=float(X_test.iloc[0]["f6"]),
    f7=float(X_test.iloc[0]["f7"]),
    f8=float(X_test.iloc[0]["f8"]),
    f9=float(X_test.iloc[0]["f9"]),
    f10=float(X_test.iloc[0]["f10"]),
    f11=float(X_test.iloc[0]["f11"])
)

response = predict_uplift(
    sample_request
)

print(response)

predicted_uplift=-0.004079329291847572 recommendation='DO_NOT_TREAT' model_name='customer_uplift_t_learner' model_alias='champion'


In [17]:
health_response = health_check()

print(
    health_response
)

{'status': 'healthy', 'model': 'customer_uplift_t_learner', 'alias': 'champion'}


## API Endpoints

The FastAPI service exposes two primary endpoints.

### GET `/health`

Used to verify that the API and registered model are available.

### POST `/predict`

Accepts the 12 model features and returns:

- Predicted uplift
- Treatment recommendation
- Registered model name
- Model alias

The prediction endpoint uses the MLflow `champion` model, allowing the deployed inference service to follow the currently assigned model version without changing the API implementation.

In [19]:
from pathlib import Path

API_PATH = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\api"
)

print("Exists:", API_PATH.exists())
print("Is file:", API_PATH.is_file())
print("Is directory:", API_PATH.is_dir())

Exists: True
Is file: True
Is directory: False


In [20]:
API_BACKUP = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\api_old"
)

API_PATH.rename(API_BACKUP)

print("Existing api file renamed to:")
print(API_BACKUP)

Existing api file renamed to:
C:\Users\ugand\customer-churn-uplift-modeling\api_old


In [21]:
API_DIR = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling\api"
)

API_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("API directory created successfully:")
print(API_DIR)

API directory created successfully:
C:\Users\ugand\customer-churn-uplift-modeling\api


In [22]:
API_FILE = (
    API_DIR / "app.py"
)

In [23]:
API_DIR = (
    PROJECT_ROOT / "api"
)

API_DIR.mkdir(
    parents=True,
    exist_ok=True
)

API_FILE = (
    API_DIR / "app.py"
)

print(
    "API directory:"
)

print(
    API_DIR
)

API directory:
C:\Users\ugand\customer-churn-uplift-modeling\api


In [24]:
api_code = r'''
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc

from pathlib import Path
from fastapi import FastAPI
from pydantic import BaseModel


PROJECT_ROOT = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling"
)

MLFLOW_DIR = (
    PROJECT_ROOT / "mlruns"
)

mlflow_tracking_path = (
    MLFLOW_DIR / "mlflow.db"
)

mlflow.set_tracking_uri(
    f"sqlite:///{mlflow_tracking_path}"
)


REGISTERED_MODEL_NAME = (
    "customer_uplift_t_learner"
)

MODEL_ALIAS = "champion"

MODEL_URI = (
    f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
)


FEATURE_COLUMNS = [
    f"f{i}"
    for i in range(12)
]


model = mlflow.pyfunc.load_model(
    MODEL_URI
)


app = FastAPI(
    title="Customer Uplift Modeling API",
    description=(
        "API for uplift prediction using "
        "the MLflow champion T-Learner."
    ),
    version="1.0.0"
)


class UpliftRequest(BaseModel):

    f0: float
    f1: float
    f2: float
    f3: float
    f4: float
    f5: float
    f6: float
    f7: float
    f8: float
    f9: float
    f10: float
    f11: float


class UpliftResponse(BaseModel):

    predicted_uplift: float
    recommendation: str
    model_name: str
    model_alias: str


def get_treatment_recommendation(
    uplift: float
) -> str:

    if uplift > 0:
        return "TREAT"

    return "DO_NOT_TREAT"


@app.get("/health")
def health_check():

    return {
        "status": "healthy",
        "model": REGISTERED_MODEL_NAME,
        "alias": MODEL_ALIAS
    }


@app.post(
    "/predict",
    response_model=UpliftResponse
)
def predict_uplift(
    request: UpliftRequest
):

    input_data = pd.DataFrame(
        [
            {
                feature: getattr(
                    request,
                    feature
                )
                for feature in FEATURE_COLUMNS
            }
        ]
    )

    prediction = model.predict(
        input_data
    )

    uplift = float(
        np.asarray(
            prediction
        ).reshape(-1)[0]
    )

    recommendation = (
        get_treatment_recommendation(
            uplift
        )
    )

    return UpliftResponse(
        predicted_uplift=uplift,
        recommendation=recommendation,
        model_name=REGISTERED_MODEL_NAME,
        model_alias=MODEL_ALIAS
    )
'''


API_FILE.write_text(
    api_code,
    encoding="utf-8"
)

print(
    "API file created:"
)

print(
    API_FILE
)

API file created:
C:\Users\ugand\customer-churn-uplift-modeling\api\app.py


In [25]:
print(
    "API file exists:",
    API_FILE.exists()
)

print(
    "API file size:",
    round(
        API_FILE.stat().st_size / 1024,
        2
    ),
    "KB"
)

API file exists: True
API file size: 2.37 KB


# Conclusion

In this notebook, the production uplift model was exposed through a **FastAPI-based inference service** using the MLflow Model Registry.

The main outcomes were:

- Loaded the `champion` T-Learner directly from the MLflow Model Registry.
- Defined the 12 model input features using a validated Pydantic request schema.
- Created a `/health` endpoint for service health verification.
- Created a `/predict` endpoint for real-time uplift prediction.
- Converted predicted uplift into a treatment recommendation.
- Tested the model and API logic locally before deployment.
- Created the production API application in `api/app.py`.
- Verified that the API application file was generated successfully.

The resulting architecture is:

Client
→ FastAPI
→ MLflow Champion Model
→ T-Learner
→ Predicted Uplift
→ Treatment Recommendation

The project has now progressed from offline experimentation to a model-serving architecture in which the API can consume the currently assigned MLflow `champion` model.

The next stage will focus on **running, testing, and containerizing the API**, making the inference service reproducible and suitable for deployment.